In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Specificare gli osservabili nella base di Pauli

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato usando i seguenti requisiti.
Si raccomanda di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
```
</details>
In meccanica quantistica, gli osservabili corrispondono a proprietà fisiche che possono essere misurate.
Considerando un sistema di spin, ad esempio, potresti essere interessato a misurare l'energia del sistema o a ottenere informazioni sull'allineamento degli spin, come la magnetizzazione o le correlazioni tra spin.

Per misurare un osservabile $O$ a $n$ qubit su un computer quantistico, devi rappresentarlo come una somma di prodotti tensoriali di operatori di Pauli, ovvero

$$
O = \sum_{k=1}^K \alpha_k P_k,~~ P_k \in {I, X, Y, Z}^{\otimes n},~~ \alpha_k \in \mathbb{R},
$$

dove

$$
I = \begin{pmatrix}
1 & 0 \\ 0 & 1
\end{pmatrix}
~~
X = \begin{pmatrix}
0 & 1 \\ 1 & 0
\end{pmatrix}
~~
Y = \begin{pmatrix}
0 & -i \\ i & 0
\end{pmatrix}
~~
Z = \begin{pmatrix}
1 & 0 \\ 0 & -1
\end{pmatrix}
$$

e si sfrutta il fatto che un osservabile è hermitiano, ovvero $O^\dagger = O$. Se $O$ non è hermitiano può comunque essere decomposto come somma di operatori di Pauli, ma il coefficiente $\alpha_k$ diventa complesso.

In molti casi, l'osservabile è naturalmente specificato in questa rappresentazione dopo aver mappato il sistema di interesse sui qubit.
Ad esempio, un sistema spin-1/2 può essere mappato su un hamiltoniano di Ising

$$
H = \sum_{\langle i, j\rangle} Z_i Z_j - \sum_{i=1}^n X_i,
$$

dove gli indici $\langle i, j\rangle$ scorrono sugli spin interagenti e gli spin sono soggetti a un campo trasversale in $X$.
L'indice in pedice indica su quale qubit agisce l'operatore di Pauli, ovvero $X_i$ applica un operatore $X$ al qubit $i$ lasciando invariati gli altri.

Nel Qiskit SDK, questo hamiltoniano può essere costruito con il seguente codice.

In [1]:
from qiskit.quantum_info import SparsePauliOp

# define the number of qubits
n = 12

# define the single Pauli terms as ("Paulis", [indices], coefficient)
interactions = [
    ("ZZ", [i, i + 1], 1) for i in range(n - 1)
]  # we assume spins on a 1D line
field = [("X", [i], -1) for i in range(n)]

# build the operator
hamiltonian = SparsePauliOp.from_sparse_list(
    interactions + field, num_qubits=n
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIIIIZZ', 'IIIIIIIIIZZI', 'IIIIIIIIZZII', 'IIIIIIIZZIII', 'IIIIIIZZIIII', 'IIIIIZZIIIII', 'IIIIZZIIIIII', 'IIIZZIIIIIII', 'IIZZIIIIIIII', 'IZZIIIIIIIII', 'ZZIIIIIIIIII', 'IIIIIIIIIIIX', 'IIIIIIIIIIXI', 'IIIIIIIIIXII', 'IIIIIIIIXIII', 'IIIIIIIXIIII', 'IIIIIIXIIIII', 'IIIIIXIIIIII', 'IIIIXIIIIIII', 'IIIXIIIIIIII', 'IIXIIIIIIIII', 'IXIIIIIIIIII', 'XIIIIIIIIIII'],
              coeffs=[ 1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,
  1.+0.j,  1.+0.j,  1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j,
 -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j])


If we would like to measure the energy the observable is the Hamiltonian itself. Alternatively, we could be
interested in measuring system properties like the average magnetization by counting the number of spins
aligned in the $Z$-direction with the observable

$$
O = \frac{1}{n} \sum_{i=1} Z_i
$$

For observables that are not given in terms of Pauli operators but in a matrix form, we first have to reformulate them
in the Pauli basis in order to evaluate them on a quantum computer.
We are always able to find such a representation as the Pauli matrices form a basis for the Hermitian $2^n \times 2^n$ matrices.
We expand the observable $O$ as

$$
O = \sum_{P \in \{I, X, Y, Z\}^{\otimes n}} \mathrm{Tr}(O P) P,
$$

where the sum runs over all possible $n$-qubit Pauli terms and $\mathrm{Tr}(\cdot)$ is the trace of a matrix, which acts as inner product.
You can implement this decomposition  from a matrix to Pauli terms using the `SparsePauliOp.from_operator` method, like so:

In [2]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp

matrix = np.array(
    [[-1, 0, 0.5, -1], [0, 1, 1, 0.5], [0.5, 1, -1, 0], [-1, 0.5, 0, 1]]
)

observable = SparsePauliOp.from_operator(matrix)
print(observable)

SparsePauliOp(['IZ', 'XI', 'YY'],
              coeffs=[-1. +0.j,  0.5+0.j,  1. -0.j])


Se volessimo misurare l'energia, l'osservabile è l'hamiltoniano stesso. In alternativa, potremmo essere
interessati a misurare proprietà del sistema come la magnetizzazione media, contando il numero di spin
allineati nella direzione $Z$ con l'osservabile

$$
O = \frac{1}{n} \sum_{i=1} Z_i
$$

Per gli osservabili non espressi in termini di operatori di Pauli ma in forma matriciale, occorre prima riformularli
nella base di Pauli per poterli valutare su un computer quantistico.
Siamo sempre in grado di trovare tale rappresentazione poiché le matrici di Pauli formano una base per le matrici hermitiane $2^n \times 2^n$.
Espandiamo l'osservabile $O$ come

$$
O = \sum_{P \in {I, X, Y, Z}^{\otimes n}} \mathrm{Tr}(O P) P,
$$

dove la somma scorre su tutti i possibili termini di Pauli a $n$ qubit e $\mathrm{Tr}(\cdot)$ è la traccia di una matrice, che funge da prodotto interno.
Puoi implementare questa decomposizione da una matrice a termini di Pauli usando il metodo `SparsePauliOp.from_operator`, come segue:

In [3]:
from qiskit.circuit import QuantumCircuit

# create a circuit, where we would like to measure
# q0 in the X basis, q1 in the Y basis and q2 in the Z basis
circuit = QuantumCircuit(3)
circuit.ry(0.8, 0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.barrier()

# diagonalize X with the Hadamard gate
circuit.h(0)

# diagonalize Y with Hadamard as S^\dagger
circuit.sdg(1)
circuit.h(1)

# the Z basis is the default, no action required here

# measure all qubits
circuit.measure_all()
circuit.draw("mpl")

<Image src="../docs/images/guides/specify-observables-pauli/extracted-outputs/ce4b1984-ebe0-44f6-a78c-d67b9e9bb361-0.svg" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
  -  Read the [SparsePauliOp API](/docs/api/qiskit/qiskit.quantum_info.SparsePauliOp#sparsepauliop) reference.
</Admonition>